# 1. Preprocessing

## 1. Setup

In [1]:
import os
import pandas as pd
import geopandas as gpd

DATA_DIR   = f'./data/raw'
OUT_DIR    = f'./data/preprocessed'
os.makedirs(OUT_DIR, exist_ok=True)

LAT_COL = 'Lat'
LON_COL = 'Long'
STATUS_COL = 'Status'

In [2]:
#pd.set_option("display.max_rows", None) # run it to list all the row
pd.reset_option("display.max_rows")

## 2. Load raw data

In [3]:
border         = gpd.read_file(f'{DATA_DIR}/OSM/milano_border.geojson')
municipalities = gpd.read_file(f'{DATA_DIR}//OSM/milano_municipalities.geojson')
nil            = gpd.read_file(f'{DATA_DIR}/OSM/milano_nil.geojson')

roads              = gpd.read_file(f'{DATA_DIR}/overture/milano_roads_overture.geojson')
places_all = gpd.read_file(f'{DATA_DIR}/overture/milano_places_overture_all.geojson')
education          = gpd.read_file(f'{DATA_DIR}/overture/milano_education_overture.geojson')
healthcare      = gpd.read_file(f'{DATA_DIR}/overture/milano_healthcare_overture.geojson')
shops           = gpd.read_file(f'{DATA_DIR}/overture/milano_shops_overture.geojson')
food_drink      = gpd.read_file(f'{DATA_DIR}/overture/milano_food_drink_overture.geojson')
squares         = gpd.read_file(f'{DATA_DIR}/overture/milano_squares_overture.geojson')

transit_stations = gpd.read_file(f'{DATA_DIR}/OpenData/mobility.geojson')
green_areas      = gpd.read_file(f'{DATA_DIR}/OpenData/green_areas.geojson')

piazze_raw = pd.read_excel(f'{DATA_DIR}/PiazzeAperte.xlsx')

nil_anagrafica = pd.read_csv(f'{DATA_DIR}/nil_anagrafica.csv')
names = ('1', 'Totale', 'Quartiere', 'Coordinate', 'NIL', '2', '3')
nil_2024       = pd.read_csv(f'{DATA_DIR}/nil_2024.csv', names=names)
names = ('1', 'densita (km quadrato)', 'Quartiere', 'Coordinate', 'NIL', '2', '3')
nil_2025       = pd.read_csv(f'{DATA_DIR}/nil_2025.csv', names=names)

print('Raw data loaded.')
print(f'  roads:            {len(roads):>6} rows')
print(f'  places (all):     {len(places_all):>6} rows')
print(f'  education:        {len(education):>6} rows')
print(f'  healthcare:       {len(healthcare):>6} rows')
print(f'  shops:            {len(shops):>6} rows')
print(f'  food & drink:     {len(food_drink):>6} rows')
print(f'  squares:          {len(squares):>6} rows')
print(f'  transit stations: {len(transit_stations):>6} rows')
print(f'  piazze aperte:    {len(piazze_raw):>6} rows')
print(f'\nPiazze Aperte columns: {piazze_raw.columns.tolist()}')
print(f'\nNIL columns: {nil_anagrafica.columns.tolist()}')
print(f'\nNIL 2024 columns: {nil_2024.columns.tolist()}')
print(f'\nNIL 2025 columns: {nil_2025.columns.tolist()}')

Raw data loaded.
  roads:             75941 rows
  places (all):      72914 rows
  education:          2725 rows
  healthcare:         2837 rows
  shops:             11381 rows
  food & drink:      11891 rows
  squares:             887 rows
  transit stations:   8635 rows
  piazze aperte:       133 rows

Piazze Aperte columns: ['Site_name', 'Address', 'NIL', 'Municipality', 'Neighbourhood', 'Urban_ring', 'Status', 'Lat', 'Long', 'School_ref']

NIL columns: ['_id', 'Anno', 'Quartiere', 'NIL', 'Uomini', 'Donne', 'Totale', 'Minori', 'Stranieri', 'Famiglie registrate in anagrafe', 'Famiglie unipersonali registrate in anagrafe', '80 e pi— soli fam registrate in anagrafe', 'Nati vivi', 'Nati fuori dal matrimonio', 'Nati con almeno un genitore straniero', 'Morti', 'Emigrati', 'Immigrati', '80 e pi—', '65 e pi—', 'Prima cittadinanza', 'residenti prima cittadinanza', 'Seconda cittadinanza', 'residenti seconda cittadinanza', 'Terza cittadinanza', 'residenti terza cittadinanza', "Scuola dell'infa

## 3. Preprocessing

### Duplicates check

In [4]:
df_list = [border, municipalities, nil, roads, places_all, education,
           healthcare, shops, food_drink, squares, transit_stations, green_areas]

df_names = ['border', 'municipalities', 'nil', 'roads', 'places_all', 'education',
            'healthcare', 'shops', 'food_drink', 'squares', 'transit_stations', 'green_areas']

In [5]:
for i, df in enumerate(df_list):
  print(df_names[i], df.geometry.duplicated().sum())

border 0
municipalities 0
nil 0
roads 0
places_all 2763
education 54
healthcare 29
shops 190
food_drink 110
squares 0
transit_stations 0
green_areas 22


In [6]:
green_areas[green_areas.geometry.duplicated(keep=False)]
# no duplicates per different categories

,area,località,category,geometry
1612,153,GIARDINO DELLA GUASTALLA,park,"MULTIPOLYGON (((9.19554 45.45966, 9.19577 45.4..."
1828,061,PARCO DI VILLA FINZI,park,"MULTIPOLYGON (((9.21919 45.50452, 9.21919 45.5..."
1830,061,PARCO DI VILLA FINZI,park,"MULTIPOLYGON (((9.21911 45.5046, 9.21919 45.50..."
1841,151,PARCO SEMPIONE,park,"MULTIPOLYGON (((9.17978 45.47485, 9.17979 45.4..."
1842,151,PARCO SEMPIONE,park,"MULTIPOLYGON (((9.18057 45.47545, 9.18057 45.4..."
1845,151,PARCO SEMPIONE,park,"MULTIPOLYGON (((9.17989 45.4763, 9.1799 45.476..."
1848,151,PARCO SEMPIONE,park,"MULTIPOLYGON (((9.17865 45.47528, 9.17883 45.4..."
2152,238,GIARDINO GONIN GIORDANI,park,"MULTIPOLYGON (((9.12318 45.44488, 9.12328 45.4..."
2153,238,GIARDINO GONIN GIORDANI,park,"MULTIPOLYGON (((9.124 45.44581, 9.124 45.44581..."
2155,238,GIARDINO GONIN GIORDANI,park,"MULTIPOLYGON (((9.12374 45.44567, 9.12386 45.4..."


In [7]:
places_all = places_all.drop_duplicates(subset="geometry").reset_index(drop=True)
shops = shops.drop_duplicates(subset="geometry").reset_index(drop=True)
food_drink = food_drink.drop_duplicates(subset="geometry").reset_index(drop=True)
education = education.drop_duplicates(subset="geometry").reset_index(drop=True)
healthcare = healthcare.drop_duplicates(subset="geometry").reset_index(drop=True)
green_areas = green_areas.drop_duplicates(subset="geometry").reset_index(drop=True)

In [8]:
print(places_all.geometry.duplicated().sum())
print(shops.geometry.duplicated().sum())
print(food_drink.geometry.duplicated().sum())
print(education.geometry.duplicated().sum())
print(healthcare.geometry.duplicated().sum())
print(green_areas.geometry.duplicated().sum())

0
0
0
0
0
0


In [9]:
transit_stations[
    transit_stations["nome"].duplicated(keep=False)
].sort_values("nome")
# we keep duplicates for geometry column as they come from different categories

,id_amat,nome,category,geometry
7296,3744.0,AULENTI GAE,bike_rack,POINT (9.18872 45.48311)
7291,3739.0,AULENTI GAE,bike_rack,POINT (9.18932 45.48324)
7292,3740.0,AULENTI GAE,bike_rack,POINT (9.18868 45.48361)
7293,3741.0,AULENTI GAE,bike_rack,POINT (9.18872 45.48369)
7295,3743.0,AULENTI GAE,bike_rack,POINT (9.18866 45.48317)
...,...,...,...,...
5549,895.0,Viale di Porta Vercellina,bike_rack,POINT (9.16559 45.46581)
6785,3120.0,Viale di Porta Vercellina,bike_rack,POINT (9.16504 45.46408)
6674,2974.0,Viale di Porta Vercellina,bike_rack,POINT (9.16464 45.46285)
4725,19863.0,Vigan� De Vizzi,bus_tram_stops,POINT (9.23586 45.56091)


In [10]:
duplicated_names = transit_stations["nome"].value_counts()
duplicated_names = duplicated_names[duplicated_names > 1].index

filtered_transit_stations = transit_stations[transit_stations['nome'].isin(duplicated_names)]

display(filtered_transit_stations.groupby(['nome', 'category']).size().reset_index(name='count').sort_values(by=['nome', 'count'], ascending=[True, False]))

,nome,category,count
0,AULENTI GAE,bike_rack,5
1,Alzaia Naviglio Grande,bike_rack,4
2,Alzaia Naviglio Pavese,bike_rack,9
3,Alzaia Naviglio Pavese,pedestrian_areas,1
4,Argonne,bus_tram_stops,2
...,...,...,...
853,Viale dell'Innovazione,bike_rack,6
854,Viale della Liberazione,bike_rack,2
855,Viale delle Rimembranze di Greco,bike_rack,2
856,Viale di Porta Vercellina,bike_rack,3


In [11]:
green_areas[
    green_areas["località"].duplicated(keep=False)
].sort_values("località")

,area,località,category,geometry
2176,264,BOSCO DI BRUZZANO,park,"MULTIPOLYGON (((9.1871 45.52674, 9.18704 45.52..."
2175,264,BOSCO DI BRUZZANO,park,"MULTIPOLYGON (((9.18597 45.52633, 9.18594 45.5..."
1741,264,BOSCO DI BRUZZANO,park,"MULTIPOLYGON (((9.18416 45.52646, 9.18417 45.5..."
2103,436,COLLINA DEI CILIEGI,park,"MULTIPOLYGON (((9.20959 45.51315, 9.20958 45.5..."
2082,436,COLLINA DEI CILIEGI,park,"MULTIPOLYGON (((9.20826 45.51073, 9.20825 45.5..."
...,...,...,...,...
247,002,vie Voltri - De Nicola,dog_park,"MULTIPOLYGON (((9.15545 45.43274, 9.15479 45.4..."
941,010,vie Voltri - Ovada,playground,"MULTIPOLYGON (((9.15673 45.43476, 9.15672 45.4..."
241,010,vie Voltri - Ovada,dog_park,"MULTIPOLYGON (((9.15545 45.43483, 9.15577 45.4..."
145,249,vie Zama - Salomone,dog_park,"MULTIPOLYGON (((9.23989 45.45109, 9.23989 45.4..."


### Add macrocategory column to places_all

In [12]:
macro_categories = {
    "education": [
        "school", "education", "elementary_school", "middle_school",
        "high_school", "preschool", "public_school", "private_school",
        "day_care_preschool", "child_care_and_day_care",
        "college_university", "library", "educational_services",
        "educational_research_institute", "vocational_and_technical_school",
        "adult_education", "nursing_school", "medical_school",
        "law_schools", "business_schools", "science_schools"
    ],

    "healthcare": [
        "pharmacy", "hospital", "doctor", "medical_center",
        "health_and_medical", "dentist", "general_dentistry",
        "pediatric_dentist", "pediatrician",
        "obstetrician_and_gynecologist", "public_health_clinic",
        "urgent_care_clinic", "womens_health_clinic",
        "childrens_hospital", "counseling_and_mental_health",
        "psychologist", "psychotherapist", "psychiatrist",
        "clinical_laboratories", "diagnostic_services",
        "eye_care_clinic", "optometrist", "medical_supply",
        "home_health_care", "retirement_home",
        "senior_citizen_services",
        "disability_services_and_support_organization",
        "physical_therapy", "medical_transportation",
        "ambulance_and_ems_services"
    ],

    "daily_food": [
        "supermarket", "grocery_store", "convenience_store",
        "bakery", "butcher_shop", "farmers_market",
        "public_market", "specialty_grocery_store",
        "organic_grocery_store", "korean_grocery_store",
        "international_grocery_store", "fruits_and_vegetables",
        "fishmonger", "seafood_market", "beverage_store",
        "food_and_beverage_store", "drugstore",
        "ethical_grocery", "cheese_shop", "delicatessen"
    ],

    "food_drink_sociality": [
        "restaurant", "italian_restaurant", "pizza_restaurant",
        "bar", "cafe", "coffee_shop", "cocktail_bar",
        "sushi_restaurant", "chinese_restaurant",
        "fast_food_restaurant", "wine_bar", "seafood_restaurant",
        "japanese_restaurant", "pub", "gelato",
        "burger_restaurant", "asian_restaurant",
        "mediterranean_restaurant", "peruvian_restaurant",
        "indian_restaurant", "vegetarian_restaurant",
        "beer_bar", "asian_fusion_restaurant",
        "barbecue_restaurant", "brazilian_restaurant",
        "american_restaurant", "mexican_restaurant",
        "european_restaurant", "turkish_restaurant",
        "gastropub", "korean_restaurant", "smoothie_juice_bar",
        "chicken_restaurant", "latin_american_restaurant",
        "vegan_restaurant", "tapas_bar", "thai_restaurant",
        "greek_restaurant", "breakfast_and_brunch_restaurant",
        "african_restaurant", "spanish_restaurant",
        "salad_bar", "bar_and_grill_restaurant",
        "piadina_restaurant", "lebanese_restaurant",
        "middle_eastern_restaurant", "theme_restaurant",
        "hotel_bar", "sports_bar", "cafeteria",
        "food_court", "food_truck", "diner", "bistro",
        "ice_cream_shop", "desserts", "bubble_tea",
        "patisserie_cake_shop", "cupcake_shop"
    ],

    "mobility": [
        "transportation", "train_station", "metro_station",
        "bus_station", "transport_interchange", "bus_service",
        "bike_rentals", "bicycle_shop", "bike_repair_maintenance",
        "ev_charging_station", "taxi_service", "parking",
        "car_sharing", "scooter_rental"
    ],

    "public_space_livability": [
        "park", "public_plaza", "playground", "dog_park",
        "fountain", "public_toilet", "public_bath_houses",
        "community_center", "cultural_center",
        "sports_and_recreation_venue", "sports_club_and_league",
        "soccer_field", "basketball_court", "tennis_court",
        "swimming_pool", "skate_park", "botanical_garden",
        "lake", "river", "canal", "garden", "urban_farm"
    ],

    "civic_social_services": [
        "post_office", "town_hall", "central_government_office",
        "local_and_state_government_offices",
        "public_service_and_government", "registry_office",
        "police_department", "fire_department", "courthouse",
        "law_enforcement", "social_service_organizations",
        "social_and_human_services", "community_services_non_profits",
        "charity_organization", "youth_organizations",
        "non_governmental_association",
        "public_and_government_association",
        "public_utility_company", "electric_utility_provider",
        "water_supplier", "recycling_center"
    ],

    "finance_basic": [
        "banks", "bank_credit_union", "atms",
        "credit_union", "money_transfer_services",
        "currency_exchange"
    ],

    "culture_recreation": [
        "museum", "art_museum", "history_museum",
        "science_museum", "modern_art_museum",
        "contemporary_art_museum", "childrens_museum",
        "art_gallery", "theatre", "cinema",
        "music_venue", "performing_arts",
        "topic_concert_venue", "stadium_arena",
        "auditorium", "landmark_and_historical_building",
        "monument", "palace",
        "arts_and_entertainment", "active_life",
        "gym", "yoga_studio", "pilates_studio",
        "martial_arts_club", "fitness_trainer"
    ],

    "retail": [
        "clothing_store", "jewelry_store", "furniture_store",
        "womens_clothing_store", "boutique", "shoe_store",
        "shopping", "flowers_and_gifts_shop", "bookstore",
        "retail", "childrens_clothing_store",
        "building_supply_store", "mobile_phone_store",
        "newspaper_and_magazines_store", "home_goods_store",
        "mens_clothing_store", "gift_shop", "antique_store",
        "tobacco_shop", "hardware_store", "toy_store",
        "home_improvement_store", "appliance_store",
        "fabric_store", "lighting_store", "bridal_shop",
        "computer_store", "shopping_center", "department_store",
        "hobby_shop", "costume_store", "outlet_store",
        "discount_store", "candy_store", "kitchen_supply_store",
        "comic_books_store", "video_game_store"
    ],

    "personal_services": [
        "beauty_salon", "hair_salon", "barber",
        "beauty_and_spa", "cosmetic_and_beauty_supplies",
        "spas", "nail_salon", "tattoo_and_piercing",
        "laundromat", "dry_cleaning", "home_cleaning",
        "massage", "massage_therapy", "skin_care",
        "day_spa", "hair_stylist", "shoe_repair",
        "sewing_and_alterations", "pet_groomer",
        "pet_services"
    ],

    "professional_business": [
        "professional_services", "event_planning",
        "real_estate_agent", "real_estate",
        "advertising_agency", "lawyer",
        "architectural_designer", "marketing_agency",
        "printing_services", "software_development",
        "corporate_office", "business_management_services",
        "insurance_agency", "financial_service",
        "interior_design", "business", "accountant",
        "legal_services", "commercial_real_estate",
        "it_service_and_computer_repair",
        "engineering_services", "telecommunications_company",
        "public_relations", "graphic_designer",
        "web_designer", "employment_agencies",
        "business_consulting", "business_to_business",
        "business_to_business_services"
    ],

    "hospitality_tourism": [
        "hotel", "accommodation", "bed_and_breakfast",
        "hostel", "resort", "travel_services",
        "travel_agents", "travel", "tours",
        "holiday_rental_home", "service_apartments",
        "luggage_storage", "sightseeing_tour_agency",
        "airport", "airport_terminal", "airport_shuttles"
    ],

    "religion": [
        "church_cathedral", "catholic_church",
        "religious_organization", "religious_school",
        "mosque", "synagogue", "temple",
        "buddhist_temple", "evangelical_church",
        "baptist_church", "pentecostal_church",
        "convents_and_monasteries"
    ]
}

In [13]:
category_to_macro = {}

for macro, cats in macro_categories.items():
    for cat in cats:
        if cat in category_to_macro:
            print(f"Warning: duplicated category: {cat}")
        category_to_macro[cat] = macro

In [14]:
places_all["macro_category"] = places_all["category"].map(category_to_macro)

places_all["macro_category"] = places_all["macro_category"].fillna("other_unclassified")

In [15]:
places_all["macro_category"].value_counts(dropna=False)

macro_category
other_unclassified         18659
professional_business      11397
food_drink_sociality        9739
retail                      6794
personal_services           4034
healthcare                  3480
culture_recreation          3341
hospitality_tourism         2568
civic_social_services       2350
daily_food                  2072
education                   1798
public_space_livability     1342
mobility                    1002
finance_basic                923
religion                     652
Name: count, dtype: int64

## 4. Clip Overture layers to Milan border

In [16]:
def clip_to_border(gdf: gpd.GeoDataFrame, border: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
    clipped = gpd.clip(gdf, border)
    print(f'  {name}: {len(gdf)} → {len(clipped)} rows after clip')
    return clipped

roads               = clip_to_border(roads,               border, 'roads')
places_all = clip_to_border(places_all, border, 'places_all')
education        = clip_to_border(education,        border, 'education')
healthcare       = clip_to_border(healthcare,       border, 'healthcare')
shops            = clip_to_border(shops,            border, 'shops')
food_drink       = clip_to_border(food_drink,       border, 'food_drink')
squares          = clip_to_border(squares,          border, 'squares')
transit_stations = clip_to_border(transit_stations, border, 'transit_stations')

roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])].copy()

  roads: 75941 → 75868 rows after clip
  places_all: 70151 → 70143 rows after clip
  education: 2671 → 2671 rows after clip
  healthcare: 2808 → 2808 rows after clip
  shops: 11191 → 11191 rows after clip
  food_drink: 11781 → 11778 rows after clip
  squares: 887 → 887 rows after clip
  transit_stations: 8635 → 7048 rows after clip


In [17]:
clip_to_border(green_areas, border, 'green_areas')

  green_areas: 2449 → 2449 rows after clip


,area,località,category,geometry
830,072,via Basso Lelio - via Saponaro,playground,"POLYGON ((9.17448 45.40554, 9.1745 45.40556, 9..."
837,267,via Saponaro,playground,"POLYGON ((9.17339 45.40591, 9.17373 45.40564, ..."
176,291,via Costantino Baroni,dog_park,"POLYGON ((9.17323 45.40779, 9.17295 45.40762, ..."
806,291,via Costantino Baroni,playground,"POLYGON ((9.17183 45.40851, 9.17181 45.40842, ..."
826,090,via Baroni n. 45,playground,"POLYGON ((9.17169 45.40876, 9.17158 45.40884, ..."
...,...,...,...,...
392,505,parco Nord,dog_park,"POLYGON ((9.20307 45.52886, 9.20307 45.52886, ..."
2175,264,BOSCO DI BRUZZANO,park,"POLYGON ((9.18594 45.5264, 9.18594 45.52642, 9..."
2176,264,BOSCO DI BRUZZANO,park,"POLYGON ((9.18704 45.52672, 9.18693 45.52667, ..."
1741,264,BOSCO DI BRUZZANO,park,"POLYGON ((9.18417 45.52646, 9.18417 45.52646, ..."


## 5. Build Piazze Aperte GeoDataFrame

In [18]:
piazze_raw = piazze_raw.dropna(subset=[LAT_COL, LON_COL]).reset_index(drop=True)
piazze_raw['piazza_id'] = piazze_raw.index

points_piazze = gpd.GeoDataFrame(
    piazze_raw,
    geometry=gpd.points_from_xy(piazze_raw[LON_COL], piazze_raw[LAT_COL]),
    crs='EPSG:4326',
)
points_piazze = gpd.clip(points_piazze, border)

points_piazze_completed = points_piazze[points_piazze[STATUS_COL] == 'completed'].copy()

print(f'Piazze Aperte (all):       {len(points_piazze)} points')
print(f'Piazze Aperte (completed): {len(points_piazze_completed)} points')
points_piazze.head()

Piazze Aperte (all):       132 points
Piazze Aperte (completed): 53 points


,Site_name,Address,NIL,Municipality,Neighbourhood,Urban_ring,Status,Lat,Long,School_ref,piazza_id,geometry
33,Piazza delle Torri Bianche,"Via Michele Saponaro / Via dei Missaglia, 2014...",41,5,Gratosoglio,3,completed,45.410366,9.174148,NaN,33,POINT (9.17415 45.41037)
35,Piazzale Fabio Chiesa,"Piazzale Fabio Chiesa, 20142, Milano, MI",42,5,Chiesa Rossa,3,completed,45.427460,9.174737,NaN,35,POINT (9.17474 45.42746)
15,Via de Nicola,"Via Enrico de Nicola / Piazza Enzo Paci, 20142...",46,6,Barona,3,completed,45.430651,9.158605,NaN,15,POINT (9.1586 45.43065)
52,Via Pescarenico,"Via Pescarenico, 20142, Milano, MI",42,5,Chiesa Rossa / Stadera,3,completed,45.432980,9.172142,NaN,52,POINT (9.17214 45.43298)
102,Via Barrili,"Via Anton Giulio Barrili, 27, 20141, Milano, MI",42,5,Stadera / Chiesa Rossa,3,proposed,45.433973,9.179837,Scuola Infanzia di Via Barrili,102,POINT (9.17984 45.43397)


## 6. Add columns in NIL anagraphical data

In [19]:
nil_anagrafica = pd.read_csv(f'{DATA_DIR}/nil_anagrafica.csv')
names = ('1', 'Totale', 'Quartiere', 'Coordinate', 'NIL', '2', '3')
nil_2024       = pd.read_csv(f'{DATA_DIR}/nil_2024.csv', names=names)
names = ('1', 'densita (km quadrato)', 'Quartiere', 'Coordinate', 'NIL', '2', '3')
nil_2025       = pd.read_csv(f'{DATA_DIR}/nil_2025.csv', names=names)

In [20]:
nil_anagrafica['Area (metri quadrati)'] = nil_anagrafica['Area (metri quadrati)'].str.replace(',', '.').astype(float)
nil_anagrafica['densita (km quadrato)'] = (nil_anagrafica['Totale'] / nil_anagrafica['Area (metri quadrati)']) * 1e6

In [21]:
nil_2024['Anno'] = 2024
nil_2025['Anno'] = 2025

nil_2024.drop(columns=['1', '2', '3'], inplace=True)
nil_2025.drop(columns=['1', '2', '3'], inplace=True)

nil_2024["NIL"] = nil_2024["NIL"].astype(str).str.replace("NIL ", "", regex=False).astype(int)
nil_2025["NIL"] = nil_2025["NIL"].astype(str).str.replace("NIL ", "", regex=False).astype(int)

for col in ["Totale"]:
    nil_2024[col] = (nil_2024[col].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
        .str.replace("-", "", regex=False).replace("", float("nan")).astype(float))
for col in ["densita (km quadrato)"]:
    nil_2025[col] = (nil_2025[col].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
        .str.replace("-", "", regex=False).replace("", float("nan")).astype(float))

area_lookup = nil_anagrafica[["Quartiere", "Area (metri quadrati)"]].drop_duplicates(subset='Quartiere')
nil_2024 = nil_2024.merge(area_lookup, on="Quartiere", how="left")
nil_2025 = nil_2025.merge(area_lookup, on="Quartiere", how="left")

nil_2024['densita (km quadrato)'] = nil_2024['Totale'] / nil_2024['Area (metri quadrati)'] * 1e6
nil_2025['Totale'] = nil_2025['densita (km quadrato)'] * nil_2025['Area (metri quadrati)'] * 1e-6


In [22]:
common_cols = nil_anagrafica.columns.intersection(nil_2024.columns).intersection(nil_2025.columns)

nil_full = pd.concat(
    [nil_anagrafica[common_cols], nil_2024[common_cols], nil_2025[common_cols]],
    ignore_index=True
)

In [23]:
nil_anagrafica.to_csv(f'{OUT_DIR}/nil_anagrafica.csv', index=False, encoding='utf-8')
nil_2024.to_csv(f'{OUT_DIR}/nil_2024.csv', index=False, encoding='utf-8')
nil_2025.to_csv(f'{OUT_DIR}/nil_2025.csv', index=False, encoding='utf-8')
nil_full.to_csv(f'{OUT_DIR}/nil_full.csv', index=False, encoding='utf-8')

## 7. Save preprocessed files

In [24]:
def save(gdf: gpd.GeoDataFrame, name: str) -> None:
    path = f'{OUT_DIR}/{name}.geojson'
    gdf.to_file(path, driver='GeoJSON')
    print(f'  Saved: {path} ({len(gdf)} features)')


save(border,                  'milano_border')
save(municipalities,          'milano_municipalities')
save(nil,                     'milano_nil')

save(roads,                   'milano_roads')
save(places_all,     'milano_places_all')
save(education,               'milano_education')
save(healthcare,              'milano_healthcare')
save(shops,                   'milano_shops')
save(food_drink,              'milano_food_drink')
save(squares,                 'milano_squares')
save(transit_stations,        'milano_transit_stations')
save(green_areas,        'milano_green_areas')

save(points_piazze,           'piazze_aperte')
save(points_piazze_completed, 'piazze_aperte_completed')

nil_anagrafica.to_csv(f'{OUT_DIR}/nil_anagrafica.csv', index=False, encoding='utf-8')
nil_2024.to_csv(f'{OUT_DIR}/nil_2024.csv', index=False, encoding='utf-8')
nil_2025.to_csv(f'{OUT_DIR}/nil_2025.csv', index=False, encoding='utf-8')
nil_full.to_csv(f'{OUT_DIR}/nil_full.csv', index=False, encoding='utf-8')

print('\nDone. All preprocessed files saved to', OUT_DIR)

  Saved: ./data/preprocessed/milano_border.geojson (1 features)
  Saved: ./data/preprocessed/milano_municipalities.geojson (9 features)
  Saved: ./data/preprocessed/milano_nil.geojson (88 features)
  Saved: ./data/preprocessed/milano_roads.geojson (75868 features)
  Saved: ./data/preprocessed/milano_places_all.geojson (70143 features)
  Saved: ./data/preprocessed/milano_education.geojson (2671 features)
  Saved: ./data/preprocessed/milano_healthcare.geojson (2808 features)
  Saved: ./data/preprocessed/milano_shops.geojson (11191 features)
  Saved: ./data/preprocessed/milano_food_drink.geojson (11778 features)
  Saved: ./data/preprocessed/milano_squares.geojson (887 features)
  Saved: ./data/preprocessed/milano_transit_stations.geojson (7048 features)
  Saved: ./data/preprocessed/milano_green_areas.geojson (2449 features)
  Saved: ./data/preprocessed/piazze_aperte.geojson (132 features)
  Saved: ./data/preprocessed/piazze_aperte_completed.geojson (53 features)

Done. All preprocessed fi